In [ ]:
import sys
!{sys.executable} -m pip install torch --index-url https://download.pytorch.org/whl/cu121

In [ ]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

In [ ]:
# Restart kernel after this completes, then skip this cell.
# sys.executable ensures packages go into the kernel's Python, not the system Python.
import sys
!{sys.executable} -m pip install transformers peft bitsandbytes accelerate -q

In [ ]:
import sys
!{sys.executable} -m pip install --upgrade transformers -q

In [ ]:
# Set these paths before running
ADAPTER_PATH      = '/path/to/lora_adapter'              # downloaded from Google Drive
JSONL_PATH        = '/path/to/synthetic_cot_dataset.jsonl'
EU_ADDITIVES_PATH = '/path/to/eu_food_additives.json'

BASE_MODEL     = 'microsoft/Phi-4-mini-instruct'
MAX_NEW_TOKENS = 1024
TEST_SIZE      = 0.2
SEED           = 42

In [ ]:
import json, re
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

ENUM_RE = re.compile(r'\bE\d{3,4}[A-Z]?\b', re.IGNORECASE)

SYSTEM_PROMPT = (
    'You are a senior Food Scientist specialized in NOVA food processing classification.\n'
    'Return STRICT JSON (no markdown) with these exact keys:\n'
    '  ingredient_steps: list of objects each with: ingredient, analysis, nova_marker, e_number, cited_function\n'
    '  reasoning_summary: string\n'
    '  predicted_nova_group: integer 1-4\n'
    'Rules:\n'
    '1) Analyse each ingredient step-by-step.\n'
    '2) When an additive appears, cite its E-number and function from the EU additive context.\n'
    '3) Be factually grounded and concise.\n'
    '4) Output only valid JSON, no markdown fences.'
)


def extract_text(raw):
    """Normalise ingredient / product-name fields from OFF parquet."""
    if not raw:
        return ''
    if isinstance(raw, dict):
        return raw.get('text', '')
    s = str(raw).strip()
    if not s.startswith('[{'):
        return s
    # Ingredient text stored as stringified list-of-dicts from OFF parquet
    key = "'text': '"
    i = s.find(key)
    if i == -1:
        return s
    start = i + len(key)
    end = s.find("'", start)
    return s[start:end] if end != -1 else s


def build_user_message(record, eu_additives):
    """Build the user-turn prompt with inline EU additive context."""
    ingredients = extract_text(record.get('ingredients_text', ''))
    product     = extract_text(record.get('product_name', 'Unknown')) or 'Unknown'
    country     = record.get('country') or 'Unknown'
    ing_lower   = ingredients.lower()

    ctx = {}

    # Match explicit E-number codes written in the ingredient text (e.g. E322)
    for code in ENUM_RE.findall(ingredients.replace('-', '')):
        code = code.upper()
        if code in eu_additives:
            ctx[code] = eu_additives[code]

    # Match by aliases (e.g. 'Lecithin' in E322 aliases matches 'Soya Lecithin')
    for code, entry in eu_additives.items():
        if code in ctx:
            continue
        for alias in entry.get('aliases', []):
            if len(alias) > 4 and alias.lower() in ing_lower:
                ctx[code] = entry
                break

    lines = []
    for code, entry in sorted(ctx.items()):
        name  = entry.get('name', '')
        funcs = [fn.replace('en:', '') for fn in entry.get('functions', [])]
        lines.append(f'{code}: name={name}, functions={funcs}')

    block = '\n'.join(lines) if lines else 'No explicit E-number found.'
    return (
        f'Product: {product}\nCountry: {country}\n'
        f'Ingredients: {ingredients}\nEU additive context:\n{block}'
    )


print('Helper functions loaded.')

In [ ]:
# Load EU additives reference database
with open(EU_ADDITIVES_PATH) as f:
    eu_additives = json.load(f)
print(f'EU additives loaded: {len(eu_additives)} entries')

# Load CoT traces
records = []
with open(JSONL_PATH) as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))
print(f'Total traces loaded: {len(records)}')

# NOVA distribution across full dataset
print('\nNOVA distribution (full dataset):')
dist = Counter(r['ground_truth_nova_group'] for r in records)
for g in sorted(dist):
    print(f'  NOVA {g}: {dist[g]} ({dist[g] / len(records) * 100:.1f}%)')

# Build conversation objects (same structure as training notebook)
def make_conversation(record):
    return {
        'messages': [
            {'role': 'system',    'content': SYSTEM_PROMPT},
            {'role': 'user',      'content': build_user_message(record, eu_additives)},
            {'role': 'assistant', 'content': json.dumps(record['trace'], ensure_ascii=False)},
        ],
        'nova_group': record['ground_truth_nova_group'],
    }

conversations = [make_conversation(r) for r in records]
labels = [c['nova_group'] for c in conversations]

# Reproduce the IDENTICAL stratified 80/20 split used during training
train_data, test_data = train_test_split(
    conversations,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=labels,
)

print(f'\nSplit sizes — Train: {len(train_data)} | Test: {len(test_data)}')
print('Train NOVA:', dict(sorted(Counter(d['nova_group'] for d in train_data).items())))
print('Test  NOVA:', dict(sorted(Counter(d['nova_group'] for d in test_data).items())))

In [ ]:
import os, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# Load tokenizer — try adapter folder first (Unsloth saves it there),
# fall back to the base model on HuggingFace if not present.
if os.path.exists(os.path.join(ADAPTER_PATH, 'tokenizer.json')):
    tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)
    print('Tokenizer loaded from adapter folder.')
else:
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    print('Tokenizer loaded from base model.')

# 4-bit config matching training
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Download and load base model in 4-bit (~8 GB download on first run)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
)

# Attach LoRA adapter
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()

print('Model + adapter loaded.')
print(f'Device: {next(model.parameters()).device}')

In [ ]:
import torch
from tqdm.auto import tqdm


def predict(conv):
    encoded = tokenizer.apply_chat_template(
        conv['messages'][:2],
        tokenize=True,
        add_generation_prompt=True,
        return_tensors='pt',
    )

    if hasattr(encoded, 'input_ids'):
        input_ids      = encoded.input_ids.to(model.device)
        attention_mask = encoded.attention_mask.to(model.device)
    else:
        input_ids      = encoded.to(model.device)
        attention_mask = torch.ones_like(input_ids)

    with torch.no_grad():
        output = model.generate(
            input_ids,
            attention_mask=attention_mask,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = tokenizer.decode(
        output[0][input_ids.shape[1]:],
        skip_special_tokens=True,
    ).strip()

    try:
        return int(json.loads(generated).get('predicted_nova_group', -1))
    except Exception:
        pass

    idx = generated.find('predicted_nova_group')
    if idx != -1:
        chunk = generated[idx: idx + 30]
        for n in (1, 2, 3, 4):
            if str(n) in chunk:
                return n

    return -1


y_true, y_pred = [], []
for conv in tqdm(test_data, desc='Evaluating'):
    y_true.append(conv['nova_group'])
    y_pred.append(predict(conv))

failures = sum(1 for p in y_pred if p == -1)
valid    = [(t, p) for t, p in zip(y_true, y_pred) if p != -1]
yt, yp   = zip(*valid) if valid else ([], [])

print(f'\nFormat failures: {failures}/{len(test_data)}')
if yt:
    acc = accuracy_score(yt, yp)
    f1  = f1_score(yt, yp, average='macro', labels=[1, 2, 3, 4], zero_division=0)
    print(f'NOVA Accuracy:   {acc:.4f}')
    print(f'Macro F1:        {f1:.4f}')
    print()
    print(classification_report(
        yt, yp,
        labels=[1, 2, 3, 4],
        target_names=['NOVA 1', 'NOVA 2', 'NOVA 3', 'NOVA 4'],
        zero_division=0,
    ))
else:
    print('No valid predictions — all outputs failed to parse.')

with open('eval_results.txt', 'w') as f:
    f.write(f'Format failures: {failures}/{len(test_data)}\n')
    if yt:
        f.write(f'NOVA Accuracy: {acc:.4f}\n')
        f.write(f'Macro F1: {f1:.4f}\n\n')
        f.write(classification_report(yt, yp, labels=[1, 2, 3, 4],
            target_names=['NOVA 1', 'NOVA 2', 'NOVA 3', 'NOVA 4'], zero_division=0))
print('Results saved to eval_results.txt')